# Assignment 5, Question 6: Data Transformation

**Points: 20**

Transform and engineer features from the clinical trial dataset.

## Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Import utilities
from q3_data_utils import load_data, clean_data, transform_types, create_bins, fill_missing

df = load_data('data/clinical_trial_raw.csv')
print(f"Loaded {len(df)} patients")

if "bmi" in df.columns:
        df.loc[df["bmi"] <= 0, "bmi"] = np.nan

if "age" in df.columns:
        df.loc[df["age"] <= 0, "age"] = np.nan

# Prewritten visualization functions for transformation analysis
def plot_distribution(series, title, figsize=(10, 6)):
    """
    Create a histogram of a numeric series.
    
    Args:
        series: pandas Series with numeric data
        title: Chart title
        figsize: Figure size tuple
    """
    plt.figure(figsize=figsize)
    series.hist(bins=30)
    plt.title(title)
    plt.xlabel('Value')
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.show()

def plot_value_counts(series, title, figsize=(10, 6)):
    """
    Create a bar chart of value counts.
    
    Args:
        series: pandas Series with value counts
        title: Chart title
        figsize: Figure size tuple
    """
    plt.figure(figsize=figsize)
    series.plot(kind='bar')
    plt.title(title)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

Loaded 10000 patients


## Part 1: Type Conversions (5 points)

1. Convert 'enrollment_date' to datetime using the `transform_types()` utility
2. Convert categorical columns ('site', 'intervention_group', 'sex') to category dtype
3. Ensure all numeric columns are proper numeric types
4. Display the updated dtypes

In [2]:
# TODO: Type conversions

df = transform_types(df, {
    'enrollment_date': 'datetime',
    'site': 'category',
    'intervention_group': 'category',
    'sex': 'category', })
# Verify type conversions
print("Data types after transformation:")
print(df.dtypes)



Data types after transformation:
patient_id                    object
age                          float64
sex                           object
bmi                          float64
enrollment_date       datetime64[ns]
systolic_bp                  float64
diastolic_bp                 float64
cholesterol_total            float64
cholesterol_hdl              float64
cholesterol_ldl              float64
glucose_fasting              float64
site                          object
intervention_group            object
follow_up_months               int64
adverse_events                 int64
outcome_cvd                   object
adherence_pct                float64
dropout                       object
dtype: object


## Part 2: Feature Engineering (8 points)

Create these new calculated columns:

1. `cholesterol_ratio` = cholesterol_ldl / cholesterol_hdl
2. `bp_category` = categorize systolic BP:
   - 'Normal': < 120
   - 'Elevated': 120-129
   - 'High': >= 130
3. `age_group` using `create_bins()` utility:
   - Bins: [0, 40, 55, 70, 100]
   - Labels: ['<40', '40-54', '55-69', '70+']
4. `bmi_category` using standard BMI categories:
   - Underweight: <18.5
   - Normal: 18.5-24.9
   - Overweight: 25-29.9
   - Obese: >=30

In [3]:
# TODO: Calculate cholesterol ratio
df['cholesterol_ratio'] = (df['cholesterol_ldl'] / df['cholesterol_hdl'])
print(df[['cholesterol_ldl', 'cholesterol_hdl', 'cholesterol_ratio']].head())




   cholesterol_ldl  cholesterol_hdl  cholesterol_ratio
0             41.0             55.0           0.745455
1            107.0             58.0           1.844828
2             82.0             56.0           1.464286
3            104.0             56.0           1.857143
4             75.0             78.0           0.961538


In [4]:
# TODO: Categorize blood pressure
df['bp_category'] = np.select(
    [
        df['systolic_bp'] < 120,
        (df['systolic_bp'] >= 120) & (df['systolic_bp'] < 130),
        (df['systolic_bp'] >= 130)
    ],
    ['Normal', 'Elevated', 'High'   
    ],
    default='Unknown'
)
print(df[['systolic_bp', 'bp_category']].head())

   systolic_bp bp_category
0        123.0    Elevated
1        139.0        High
2        123.0    Elevated
3        116.0      Normal
4         97.0      Normal


**Note:** The `create_bins()` function has an optional `new_column` parameter. If you don't specify it, the new column will be named `{original_column}_binned`. You can use `new_column='age_group'` to give it a custom name.


In [5]:
# TODO: Create age groups
df = create_bins(
    df,
    column = "age",
    bins=[0, 40, 55, 70, 100],
    labels=['<40', '40-54', '55-69', '70+'],
    new_column='age_group'
)
print(df[['age', 'age_group']].head())

    age age_group
0  80.0       70+
1  80.0       70+
2  82.0       70+
3  95.0       70+
4  95.0       70+


In [6]:
# TODO: Create BMI categories
df['bmi_category'] = np.select(
    [
        df['bmi'] < 18.5,
        (df['bmi'] >= 18.5) & (df['bmi'] < 24.9),
        (df['bmi'] >= 25) & (df['bmi'] < 29.9),
        df['bmi'] >= 30
    ],
    ['Underweight', 'Normal weight', 'Overweight', 'Obesity'],
    default='Unknown'
)
print(df[['bmi', 'bmi_category']].head())

    bmi bmi_category
0  29.3   Overweight
1   NaN      Unknown
2   NaN      Unknown
3  25.4   Overweight
4   NaN      Unknown


## Part 3: String Cleaning (2 points)

If there are any string columns that need cleaning:
1. Convert to lowercase
2. Strip whitespace
3. Replace any placeholder values

In [7]:
# TODO: String cleaning
# Clean and standardize site names and intervention groups
for col in ["site", "intervention_group"]:
        df[col] = (
         df[col]
            .str.strip()
            .str.replace("_", " ")
            .str.replace(r"\s+", " ", regex=True)
            .str.title()
    )

# Apply custom fixes for intervention_group
df["intervention_group"] = (
    df["intervention_group"]
    .str.replace("Treatmenta", "Treatment A")
    .str.replace("Treatmen A", "Treatment A")
    .str.replace("Contrl", "Control")
)

df["sex"] = (
    df["sex"].astype(str)
              .str.strip()
              .str[0]            
              .str.upper()
              .map({"M": "Male", "F": "Female"})
)



print(df[['site', 'intervention_group', 'sex']].head())

     site intervention_group     sex
0  Site B            Control  Female
1  Site A            Control  Female
2  Site C        Treatment B  Female
3  Site D        Treatment B  Female
4  Site E        Treatment A    Male


## Part 4: One-Hot Encoding (5 points)

Create dummy variables for categorical columns:
1. One-hot encode 'intervention_group' using `pd.get_dummies()`
2. One-hot encode 'site'
3. Drop the original categorical columns
4. Show the new shape and column names

In [8]:
# TODO: One-hot encoding

cols = ["intervention_group", "site"]
encoded = pd.get_dummies(df, columns=cols, prefix=cols, drop_first=False)
print(df.filter(like='site_').head())


print("New Shape:", encoded.shape)
print("Columns:", encoded.columns.tolist())



Empty DataFrame
Columns: []
Index: [0, 1, 2, 3, 4]
New Shape: (10000, 28)
Columns: ['patient_id', 'age', 'sex', 'bmi', 'enrollment_date', 'systolic_bp', 'diastolic_bp', 'cholesterol_total', 'cholesterol_hdl', 'cholesterol_ldl', 'glucose_fasting', 'follow_up_months', 'adverse_events', 'outcome_cvd', 'adherence_pct', 'dropout', 'cholesterol_ratio', 'bp_category', 'age_group', 'bmi_category', 'intervention_group_Control', 'intervention_group_Treatment A', 'intervention_group_Treatment B', 'site_Site A', 'site_Site B', 'site_Site C', 'site_Site D', 'site_Site E']


## Part 5: Save Transformed Data

Save the fully transformed dataset to `output/q6_transformed_data.csv`

In [9]:
# TODO: Save transformed data
df_transformed_encoded = encoded
df_transformed = df
df_transformed_encoded.to_csv('output/q6_transformed_data_encoded.csv', index=False)
df_transformed.to_csv('output/q6_transformed_data.csv', index=False)
